In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip dataset.zip
!pip install opencv-python scikit-learn numpy

In [ ]:
import cv2
import numpy as np

def extract_features(video_path):
    cap = cv2.VideoCapture(video_path)

    ret, prev = cap.read()
    if not ret:
        return None

    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

    motion_magnitudes = []
    motion_directions = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        motion_magnitudes.append(np.mean(magnitude))
        motion_directions.append(np.std(angle))

        prev_gray = gray

    cap.release()

    if len(motion_magnitudes) == 0:
        return None

    features = [
        np.mean(motion_magnitudes),
        np.std(motion_magnitudes),
        np.mean(motion_directions),
        np.std(motion_directions)
    ]

    return features

In [ ]:
import cv2
import numpy as np

def extract_features(video_path):
    cap = cv2.VideoCapture(video_path)

    ret, prev = cap.read()
    if not ret:
        return None

    prev = cv2.resize(prev, (320, 240))  # resize (FAST)
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

    motion_magnitudes = []
    motion_directions = []

    frame_count = 0
    skip = 5   # 🔥 process every 5th frame

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        if frame_count % skip != 0:
            continue

        frame = cv2.resize(frame, (320, 240))  # resize
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            0.5, 2, 10, 2, 3, 1.1, 0
        )

        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        motion_magnitudes.append(np.mean(magnitude))
        motion_directions.append(np.std(angle))

        prev_gray = gray

    cap.release()

    if len(motion_magnitudes) == 0:
        return None

    features = [
        np.mean(motion_magnitudes),
        np.std(motion_magnitudes),
        np.mean(motion_directions),
        np.std(motion_directions)
    ]

    return features

In [ ]:
import os
import numpy as np

X = []
y = []

dataset_path = "dataset"

labels = {
    "healthy video": 0,
    "diseased video": 1
}

for folder in ["healthy video", "diseased video"]:
    folder_path = os.path.join(dataset_path, folder)

    for file in os.listdir(folder_path):
        video_path = os.path.join(folder_path, file)

        print("Processing:", video_path)

        features = extract_features(video_path)

        if features is not None:
            X.append(features)
            y.append(labels[folder])

X = np.array(X)
y = np.array(y)

print("Dataset size:", X.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

In [ ]:
def predict(video_path):
    features = extract_features(video_path)

    if features is None:
        return "Error in video"

    features = np.array(features).reshape(1, -1)
    pred = model.predict(features)

    return "Diseased 🦆" if pred[0] == 1 else "Healthy 🦆"

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
test_video = list(uploaded.keys())[0]
print("Result:", predict(test_video))

In [ ]:
import joblib

joblib.dump(model, "duck_disease_model.pkl")

In [ ]:
from google.colab import files
files.download("duck_disease_model.pkl")